# 01 — Prepare the wallet random-forest cohort

This add-on follows the supplied logistic-regression project's **conditional gross-profit** task. Each row is a wallet–market pair. It reuses the original features, conservative event-group unions, recorded-resolution maturity checks and exact outer split membership.

**Run order:** this notebook → `02_rf_models.ipynb` → `03_rf_validation.ipynb`.

The combined package includes the two-market pilot and market summaries, but **not the complete full-week wallet data**. Point `PROJECT_ROOT` below to the complete original project. This notebook stops with a specific missing-file message instead of substituting synthetic data.

All historical split boundaries below use **UTC**, matching the original notebooks. October 4–5 is a previously inspected test period; new results remain exploratory.

## 1. Imports and folder setup

Install dependencies once in your notebook environment: `python -m pip install -r requirements.txt`. The shared helper is ordinary Python source beside the notebooks, so every formula is available for review.

In [1]:
# Locate this add-on folder whether Jupyter starts here or in 02_notebooks.
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image

here = Path.cwd().resolve()
helper_dir = next((p for base in [here, *here.parents]
                   for p in [base, base / '02_notebooks']
                   if (p / 'rf_common.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('Keep rf_common.py beside these notebooks and start Jupyter in the extracted add-on.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
PACKAGE_ROOT = helper_dir.parent
import rf_common as rf
print('Add-on folder:', PACKAGE_ROOT)

Add-on folder: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis


## 2. Set the original project path and resource limits

Edit `PROJECT_ROOT`. `VALIDATION_DIR` can normally remain `None`: the loader looks for a complete matching set of original history/split/event exports. If it finds different runs, choose a directory explicitly.

`REBUILD_FROM_RAW=True` can generate those exports using **only the data-preparation cells** of the included original notebook 03. It needs the original candidate parquet, metadata CSV and seven weekly trade parquets; it does not fit the original logistic models. It can take substantial time and disk space.

Default tuning uses eight parameter settings, 200 trees, at most four CPU workers, and two forward daily CV folds. These settings are frozen before test evaluation. RF uses no oversampling or class weights by default, matching the logistic task's natural prevalence.

In [2]:
from datetime import datetime, timezone

# EDIT THIS: the folder containing the ORIGINAL project's 02_notebooks and 04_results.
project_paths = json.loads((PACKAGE_ROOT / 'project_paths.json').read_text()) if (PACKAGE_ROOT / 'project_paths.json').is_file() else {}
PROJECT_ROOT = Path(os.environ.get('WALLET_PROJECT_ROOT', project_paths.get('full_project_root', str(PACKAGE_ROOT)))).expanduser().resolve()
# Example on your Mac:
# PROJECT_ROOT = Path('/Users/asuka/Downloads/wallet_profit_research')

VALIDATION_DIR = os.environ.get('WALLET_VALIDATION_BUNDLE') or None
CANDIDATE_PATH = os.environ.get('WALLET_CANDIDATE_PATH') or None
REBUILD_FROM_RAW = project_paths.get('rebuild_from_raw', False)

OUTPUT_BASE = Path(os.environ.get('WALLET_RF_OUTPUT_BASE', str(PACKAGE_ROOT / '04_results' / 'random_forest'))).expanduser().resolve()
RUN_NAME = os.environ.get('WALLET_RF_RUN_NAME', datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ'))
RUN_DIR = OUTPUT_BASE / RUN_NAME
CONFIG = dict(rf.DEFAULTS)

# Optional resource changes, made BEFORE fitting:
# CONFIG.update(n_candidates=4, n_estimators=100, n_jobs=2)

# Internal executable smoke checks use this environment flag; real runs leave it unset.
if os.environ.get('WALLET_RF_SMOKE') == '1':
    CONFIG.update(n_candidates=2, n_estimators=12, n_jobs=1,
                  min_cv_train=20, min_cv_validation=20, permutation_repeats=2,
                  permutation_max_rows=160, bootstrap_replicates=20,
                  evaluation_status='SYNTHETIC SMOKE DATA — not wallet research results')

if (RUN_DIR / 'data_manifest.json').exists():
    raise FileExistsError('This run is already prepared. Continue with notebook 02, or choose a new RUN_NAME.')
print('Original project:', PROJECT_ROOT)
print('New output directory:', RUN_DIR)
display(CONFIG)

Original project: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis
New output directory: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis/04_results/random_forest/full_data_run
{'seed': 20261019, 'validation_start': '2026-10-03T00:00:00Z', 'test_start': '2026-10-04T00:00:00Z', 'full_window_start': '2026-09-30T00:00:00Z', 'week_start': '2026-09-29T00:00:00Z', 'n_candidates': 8, 'n_estimators': 200, 'n_jobs': 4, 'min_cv_train': 100, 'min_cv_validation': 100, 'cv_boundaries': ['2026-10-01T00:00:00Z', '2026-10-02T00:00:00Z', '2026-10-03T00:00:00Z'], 'permutation_repeats': 5, 'permutation_max_rows': 5000, 'bootstrap_replicates': 300, 'threshold': 0.5, 'evaluation_status': 'Previously inspected test period; exploratory reuse'}


## 3. Find or rebuild the original feature/split exports

The preferred route uses three exports **from the same run**: `fixed24h_history_features.parquet`, `row_split_audit.csv`, and `official_event_connected_mapping.csv`. Row membership comes from the original audit; no new random split is introduced.

If the original files are only symlinks, their targets must exist. A code-only ZIP cannot replace those datasets.

In [3]:
bundle = rf.validation_bundle(PROJECT_ROOT, VALIDATION_DIR)
if bundle is None and REBUILD_FROM_RAW:
    bundle = rf.rebuild_original_preparation(
        PROJECT_ROOT, RUN_DIR,
        PACKAGE_ROOT / 'reference' / 'original_03_final_validation.ipynb')
elif bundle is None:
    raise FileNotFoundError(
        'Original validation exports are missing. Set PROJECT_ROOT/VALIDATION_DIR to the full project, '
        'run its original notebook 03, or set REBUILD_FROM_RAW=True when raw inputs are available. '
        'The attached ZIP contains no data.')
print('Original validation exports:', bundle)

Original validation exports: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis/04_results/03_history_validation


## 4. Join features and verify exact matched cohorts

The loader verifies unique keys, matching entry timestamps, event-group agreement, both split and resolution boundaries, official-event separation where available, and strictly prior history. Every variant uses only rows with full provided-file 24-hour coverage.

It saves separate development-train, outer-validation, final-refit and test tables. Wallet IDs and market IDs are retained only for audits, never used as predictors.

In [4]:
frames, manifest = rf.prepare_data(PROJECT_ROOT, RUN_DIR, bundle, CANDIDATE_PATH)
rf.dump_json(RUN_DIR / 'run_config.json', CONFIG)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
rf.dump_json(OUTPUT_BASE / 'latest_run.json', {'run_dir': str(RUN_DIR)})
display(rf.cohort_summary(frames))
print('Source candidate rows:', manifest['candidate_rows'])
print('Prepared real-data run:', RUN_DIR)

       cohort  ...                                 sorted_keys_sha256
0       train  ...  b02bbd743b187024751cdc6739f5e1cdc5274136406e3d...
1  validation  ...  4fe65ac6e4e8ca4a8f3dc96c38c9a155c6b2479ccbef30...
2        test  ...  8d139fc40610e3860ea5f2c670ba28cf8d18f95770252e...
3       refit  ...  8fe82d475113167faee5b77dcb3349fea9b3272905afb3...

[4 rows x 10 columns]
Source candidate rows: 96184
Prepared real-data run: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis/04_results/random_forest/full_data_run


## 5. Inspect the five feature variants

`expanding` means supplied-week-to-date history starting September 29. `fixed24h` means `[T−24 hours, T)`. Both exclude the target market. The three history concepts are prior trades, prior markets and prior dollars bought. The five-concept versions add buy-market concentration and average buy size, plus the original missing-value indicators.

Baseline transformations, including `p`, `p²`, `p³`, are retained exactly for comparison. They are not required by a forest, but changing them here would change the experimental feature set.

In [5]:
import pandas as pd
features = pd.DataFrame([
    {'variant': variant, 'columns': rf.prediction_design(frames['train'], variant).shape[1],
     'feature_names': ', '.join(rf.prediction_design(frames['train'], variant).columns)}
    for variant in rf.VARIANTS])
features.to_csv(RUN_DIR / 'feature_sets.csv', index=False)
display(features)
print('Next: open 02_rf_models.ipynb and Run All.')

       variant  columns                                      feature_names
0     baseline       13  entry_log_buy, entry_price, entry_price_square...
1  expanding_3       16  entry_log_buy, entry_price, entry_price_square...
2   fixed24h_3       16  entry_log_buy, entry_price, entry_price_square...
3  expanding_5       20  entry_log_buy, entry_price, entry_price_square...
4   fixed24h_5       20  entry_log_buy, entry_price, entry_price_square...
Next: open 02_rf_models.ipynb and Run All.


## Interpretation boundaries

- The accounting label is conditional gross profitability, not verified net profit or insider status.
- Recorded settlement time is the available maturity proxy. Actual historical publication times remain unverified.
- The original event grouping is preserved; repeated wallets can occur in different events.
- This one-week candidate universe and the reused test dates limit generalization. New forward CV does not make the original test fresh.

These notebooks implement the random-forest comparison. They do not reinterpret or reproduce the original fixed-effects association p-values.